# Runoff Forecasting (3-Day Ahead)

Step-by-step workflow from data preparation to Random Forest evaluation.

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

ModuleNotFoundError: No module named 'pandas'

## Step 1 - Load and Prepare Data

In [ ]:
# Load dataset
df = pd.read_excel("datasets/Kasol.xlsx", sheet_name="Sheet1")

# Convert DATE to datetime
df["DATE"] = pd.to_datetime(df["DATE"])

# Sort by date (very important for time series)
df = df.sort_values("DATE").reset_index(drop=True)

# Quick check
print(df.head())
print(df.shape)

## Step 2-7 - Build Target and Features

In [ ]:
# Step 2: Create 3-day ahead target
df["Target_t_plus_3"] = df["Discharge (CUMEC)"].shift(-3)

# Drop last 3 rows (no future value available)
df = df.dropna().reset_index(drop=True)

# Step 3: Add discharge lags
df["Discharge_t-1"] = df["Discharge (CUMEC)"].shift(1)
df["Discharge_t-2"] = df["Discharge (CUMEC)"].shift(2)
df["Discharge_t-3"] = df["Discharge (CUMEC)"].shift(3)

# Step 4: Rainfall memory features
df["PCP_rolling_3"] = df["PCP"].rolling(window=3).sum()
df["PCP_rolling_7"] = df["PCP"].rolling(window=7).sum()
df["P1_rolling_3"] = df["P1"].rolling(window=3).sum()
df["P1_rolling_7"] = df["P1"].rolling(window=7).sum()
df["P2_rolling_3"] = df["P2"].rolling(window=3).sum()
df["P2_rolling_7"] = df["P2"].rolling(window=7).sum()
df["P3_rolling_3"] = df["P3"].rolling(window=3).sum()
df["P3_rolling_7"] = df["P3"].rolling(window=7).sum()

# Step 5: Temperature features
df["Temp_mean"] = (df["TMAX"] + df["TMIN"]) / 2
df["Temp_range"] = df["TMAX"] - df["TMIN"]

# Step 6: Seasonal encoding
day_of_year = df["DATE"].dt.dayofyear
df["sin_doy"] = np.sin(2 * np.pi * day_of_year / 365)
df["cos_doy"] = np.cos(2 * np.pi * day_of_year / 365)

# Step 7: Drop NaNs created by lags/rolling windows
df = df.dropna().reset_index(drop=True)

print("Final dataset shape:", df.shape)
df.head()

## Step 8 - Select Features and Prepare X, y

In [ ]:
# Select input features
feature_cols = [
    "PCP",
    "P1",
    "P2",
    "P3",
    "PCP_rolling_3",
    "PCP_rolling_7",
    "P1_rolling_3",
    "P1_rolling_7",
    "P2_rolling_3",
    "P2_rolling_7",
    "P3_rolling_3",
    "P3_rolling_7",
    "TMAX",
    "TMIN",
    "Temp_mean",
    "Temp_range",
    "rh",
    "solar",
    "wind ",
    "Discharge_t-1",
    "Discharge_t-2",
    "Discharge_t-3",
    "sin_doy",
    "cos_doy"
]

X = df[feature_cols]
y = df["Target_t_plus_3"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

## Step 9-13 - Split, Train, Predict, Evaluate

In [ ]:
# Step 9: Time-based train/test split
df["Year"] = df["DATE"].dt.year

# Train: 1979-2000
train = df[df["Year"] <= 2000]

# Test: 2006-2009
test = df[df["Year"] >= 2006]

X_train = train[feature_cols]
y_train = train["Target_t_plus_3"]
X_test = test[feature_cols]
y_test = test["Target_t_plus_3"]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

# Step 10: Train Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Step 11: Make predictions
y_pred = rf_model.predict(X_test)

# Step 12: Evaluate performance
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

# Step 13: Nash-Sutcliffe Efficiency
def nse(obs, pred):
    return 1 - (np.sum((obs - pred)**2) / np.sum((obs - np.mean(obs))**2))

nse_value = nse(y_test.values, y_pred)
print("NSE:", nse_value)

In [ ]:
# Feature importance
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(10)

## Step 14 - Extreme Event Evaluation (Top 10%)

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Define extreme threshold (top 10%)
threshold = np.percentile(y_test, 90)
extreme_mask = y_test >= threshold

y_test_extreme = y_test[extreme_mask]
y_pred_extreme = y_pred[extreme_mask]

rmse_extreme = np.sqrt(mean_squared_error(y_test_extreme, y_pred_extreme))

def nse(obs, pred):
    return 1 - (np.sum((obs - pred)**2) /
                np.sum((obs - np.mean(obs))**2))

nse_extreme = nse(y_test_extreme.values, y_pred_extreme)

print("Extreme RMSE:", rmse_extreme)
print("Extreme NSE:", nse_extreme)

## Step 15 - Visual Check (First 200 Test Days)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:200], label="Observed")
plt.plot(y_pred[:200], label="Predicted")
plt.legend()
plt.title("Observed vs Predicted Discharge (First 200 Test Days)")
plt.show()

## Step 16 - LSTM Sequence Setup (Window 14, Horizon 3)

In [ ]:
# Decide sequence structure
window_size = 14
horizon = 3

# Select features for LSTM (no lag columns)
lstm_features = [
    "Discharge (CUMEC)",
    "PCP",
    "P1",
    "P2",
    "P3",
    "TMAX",
    "TMIN",
    "rh",
    "solar",
    "wind ",
    "sin_doy",
    "cos_doy"
]

# Scale data for LSTM
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[lstm_features])
scaled_df = pd.DataFrame(scaled_data, columns=lstm_features)

## Step 17 - Create LSTM Sequences

In [ ]:
X_seq = []
y_seq = []

for i in range(window_size, len(scaled_df) - horizon):
    X_seq.append(scaled_df.iloc[i-window_size:i].values)
    y_seq.append(df["Discharge (CUMEC)"].iloc[i + horizon])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print("X shape:", X_seq.shape)
print("y shape:", y_seq.shape)

## Step 18 - Time-Based Sequence Split

In [ ]:
# Align dates with sequences
sequence_dates = df["DATE"].iloc[window_size:len(df)-horizon].reset_index(drop=True)

# Create masks
train_mask = sequence_dates.dt.year <= 2000
test_mask = sequence_dates.dt.year >= 2006

X_train = X_seq[train_mask]
y_train = y_seq[train_mask]
X_test = X_seq[test_mask]
y_test = y_seq[test_mask]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## Step 6 - Build LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(14, 12)))
model.add(Dropout(0.2))

model.add(LSTM(32))
model.add(Dropout(0.2))

model.add(Dense(16, activation='relu'))
model.add(Dense(1))

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

## Step 7 - Train LSTM with Early Stopping

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Scale target using training set only
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1))

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train_scaled,
    validation_split=0.1,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print("Epochs trained:", len(history.history['loss']))
print("EarlyStopping triggered:", len(history.history['loss']) < 50)
print("Best val loss:", min(history.history['val_loss']))
print("Final train loss:", history.history['loss'][-1])
print("Final val loss:", history.history['val_loss'][-1])

## Step 8-10 - Predict and Evaluate (Overall + Extreme)

In [ ]:
# Step 8: Make predictions
y_pred_scaled = model.predict(X_test)
y_pred_lstm = y_scaler.inverse_transform(y_pred_scaled).flatten()

# Step 9: Evaluate performance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse_lstm = np.sqrt(mean_squared_error(y_test, y_pred_lstm))
mae_lstm = mean_absolute_error(y_test, y_pred_lstm)
r2_lstm = r2_score(y_test, y_pred_lstm)

def nse(obs, pred):
    return 1 - (np.sum((obs - pred)**2) /
                np.sum((obs - np.mean(obs))**2))

nse_lstm = nse(y_test, y_pred_lstm)

print("LSTM RMSE:", rmse_lstm)
print("LSTM R2:", r2_lstm)
print("LSTM NSE:", nse_lstm)

# Step 10: Extreme event evaluation
threshold = np.percentile(y_test, 90)
mask = y_test >= threshold

y_test_ext = y_test[mask]
y_pred_ext = y_pred_lstm[mask]

rmse_ext = np.sqrt(mean_squared_error(y_test_ext, y_pred_ext))
nse_ext = nse(y_test_ext, y_pred_ext)

print("LSTM Extreme RMSE:", rmse_ext)
print("LSTM Extreme NSE:", nse_ext)